In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 날씨 데이터 전처리

In [ ]:
df = pd.read_csv(folder_path + "NewYork_weather_170830_181231.csv")

In [ ]:
df.columns

In [ ]:
import glob
import os

folder_path = "/content/drive/My Drive/베이지안자료분석PBL/data/raw_data/weather_data/"

# 폴더 내 .csv 파일 목록 가져오기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

dataframes = []

for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

weather_data = pd.concat(dataframes, ignore_index=True)

weather_data = weather_data[["datetime", "temp", "humidity", "cloudcover", "solarradiation", "preciptype"]].rename(
    columns={"datetime" : "date", "cloudcover": "cloud_cover", "solarradiation" : "solar_radiation", "preciptype" : "is_rain"}
)
weather_data["date"] = pd.to_datetime(weather_data["date"])

weather_data.sort_values(by="date", inplace=True)
weather_data = weather_data[:-2]

weather_data.head(3)

In [ ]:
# Update the 'is_rain' column: Replace NaN with 0, non-NaN with 1
weather_data['is_rain'] = weather_data['is_rain'].notna().astype(int)

# Display the first few rows to verify the changes
weather_data[['is_rain']].head()

In [ ]:
weather_data.to_csv("/content/drive/My Drive/베이지안자료분석PBL/data/pre_data/weather_data.csv", index=False)

In [ ]:
folder_path = "/content/drive/My Drive/베이지안자료분석PBL/data/raw_data/"

btc_data = pd.read_csv(folder_path + "btc_data.csv")
btc_data["date"] = pd.to_datetime(btc_data["date"])

In [ ]:
btc_data

In [ ]:
len(weather_data)

In [ ]:
len(btc_data)

In [ ]:
data = pd.merge(btc, weather, on='date', how='outer')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.hist(data['close_pct'], bins=50)
plt.xlabel('close_pct')
plt.ylabel('Frequency')
plt.title('Histogram of close_pct')
plt.show()

In [ ]:
cols = ['mfi', 'williams', 'temp', 'humidity', 'cloud_cover', 'solar_radiation']

for col in cols:
  plt.hist(data[col], bins=50)
  plt.xlabel('close_pct')
  plt.ylabel('Frequency')
  plt.title(f'Histogram of {col}')
  plt.show()


In [ ]:
data.columns

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, AutoDateLocator

In [ ]:
data["date"] = pd.to_datetime(data["date"])
x = data["date"]
y1 = data['future_close_pct']

plt.figure(figsize=(12, 6))
plt.plot(x, y1, marker='o', ms = 4, linestyle='-', color='r', lw = 0.5)

ax = plt.gca()
ax.xaxis.set_major_locator(AutoDateLocator())
ax.xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))

plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

df = data.copy()
independent_variables = [
       'temp', 'humidity', 'cloud_cover', 'solar_radiation']

X = df[independent_variables]
y = df['future_close_pct']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # 스케일링된 X 반환

# 스케일링 결과를 데이터프레임으로 변환
X_scaled = pd.DataFrame(X_scaled, columns=independent_variables)

# 상수항 추가
X_scaled = sm.add_constant(X_scaled)
model = sm.OLS(y, X_scaled).fit()
residuals = model.resid

lags = list(range(1, 7))
lagged_residuals = pd.concat([residuals.shift(lag) for lag in lags], axis=1)
lagged_residuals.columns = [f'resid_lag_{lag}' for lag in lags]
lagged_residuals = lagged_residuals.dropna()

valid_index = lagged_residuals.index
y = y.loc[valid_index]
X_scaled_combined = pd.concat([X_scaled.loc[valid_index], lagged_residuals], axis=1)

combined_model = sm.OLS(y, X_scaled_combined)
results = combined_model.fit()

print(results.summary())